In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake session connected successfully!")

In [ ]:
result = session.sql("""
    SELECT
        CURRENT_DATABASE() AS DATABASE_NAME,
        CURRENT_SCHEMA() AS SCHEMA_NAME,
        CURRENT_ROLE() AS ROLE_NAME,
        CURRENT_WAREHOUSE() AS WAREHOUSE_NAME
""")

result.show()

In [ ]:
session.sql("SHOW DATABASES").show()

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, upper

session = get_active_session()

policies = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.POLICIES"
)

active_policies = policies.filter(
    upper(col("POLICY_STATUS")) == "ACTIVE"
)

active_count = active_policies.count()

print("KPI: ACTIVE POLICIES")
print("Total Active Policies:", active_count)

In [ ]:
from snowflake.snowpark.functions import col, upper, sum as sum_

policies = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.POLICIES"
)

active_policies = policies.filter(
    upper(col("POLICY_STATUS")) == "ACTIVE"
)

result = active_policies.select(
    sum_(col("SUM_INSURED")).alias("TOTAL_SUM_INSURED")
)

print("KPI: SUM INSURED")
result.show()

In [ ]:
from snowflake.snowpark.functions import col, upper, sum as sum_

payments = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.PREMIUM_PAYMENTS"
)

paid_payments = payments.filter(
    upper(col("PAYMENT_STATUS")) == "PAID"
)

result = paid_payments.select(
    sum_(col("PAYMENT_AMOUNT")).alias("TOTAL_PREMIUM_COLLECTED")
)

print("KPI: PREMIUM COLLECTED")
result.show()

In [ ]:
session.sql("""
    SHOW TABLES IN SCHEMA
    INSURANCEPOLICY_LIFECYCLE_DB.SILVER
""").show()

In [ ]:
claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

In [ ]:
from snowflake.snowpark.functions import col, upper

policies = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.POLICIES"
)

claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

active_count = policies.filter(
    upper(col("POLICY_STATUS")) == "ACTIVE"
).count()

total_claims = claims.count()

claim_frequency = (
    (total_claims / active_count) * 100
    if active_count > 0 else 0
)

print("KPI: CLAIM FREQUENCY")
print("Total Claims:", total_claims)
print("Active Policies:", active_count)
print("Claim Frequency:", round(claim_frequency, 2), "%")

In [ ]:
from snowflake.snowpark.functions import col, avg

claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

# Consider only current SCD Type 2 records
current_claims = claims.filter(
    col("IS_CURRENT") == True
)

result = current_claims.select(
    avg(col("CLAIM_AMOUNT")).alias("AVERAGE_CLAIM_AMOUNT")
)

print("KPI: AVERAGE CLAIM AMOUNT")
result.show()

In [ ]:
from snowflake.snowpark.functions import col, upper

claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

current_claims = claims.filter(
    col("IS_CURRENT") == True
)

total_claims = current_claims.count()

approved_claims = current_claims.filter(
    upper(col("CLAIM_STATUS")) == "APPROVED"
).count()

approval_ratio = (
    (approved_claims / total_claims) * 100
    if total_claims > 0 else 0
)

print("KPI: APPROVAL RATIO")
print("Total Claims:", total_claims)
print("Approved Claims:", approved_claims)
print("Approval Ratio:", round(approval_ratio, 2), "%")

In [ ]:
claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

settlements = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIM_SETTLEMENTS"
)

print("CLAIMS_SCD2 columns:")
claims.print_schema()

print("CLAIM_SETTLEMENTS columns:")
settlements.print_schema()

In [ ]:
from snowflake.snowpark.functions import (
    col, avg, datediff
)

claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

settlements = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIM_SETTLEMENTS"
)

# Get only current claim records
current_claims = claims.filter(
    (col("IS_CURRENT") == True) &
    (col("IS_DELETED") == False)
)

# Select required columns and rename them
claim_dates = current_claims.select(
    col("CLAIM_ID").alias("C_ID"),
    col("CLAIM_REGISTRATION_DATE").alias("REG_DATE")
)

settlement_dates = settlements.select(
    col("CLAIM_ID").alias("S_ID"),
    col("SETTLEMENT_DATE").alias("SET_DATE")
)

# Join using matching claim IDs
joined_data = claim_dates.join(
    settlement_dates,
    claim_dates["C_ID"] == settlement_dates["S_ID"],
    "inner"
)

# Calculate turnaround days
turnaround_data = joined_data.with_column(
    "TURNAROUND_DAYS",
    datediff(
        "day",
        col("REG_DATE"),
        col("SET_DATE")
    )
)

# Remove invalid negative durations
valid_turnaround = turnaround_data.filter(
    col("TURNAROUND_DAYS") >= 0
)

# Calculate average turnaround
result = valid_turnaround.select(
    avg(col("TURNAROUND_DAYS")).alias(
        "AVERAGE_SETTLEMENT_TURNAROUND_DAYS"
    )
)

print("KPI 7: SETTLEMENT TURNAROUND TIME")
result.show()

In [ ]:
from snowflake.snowpark.functions import col, sum as sum_

settlements = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIM_SETTLEMENTS"
)

result = settlements.select(
    sum_(col("NET_SETTLEMENT_AMOUNT")).alias(
        "TOTAL_NET_SETTLEMENT_AMOUNT"
    )
)

print("KPI 8: NET SETTLEMENT AMOUNT")
result.show()

In [ ]:
from snowflake.snowpark.functions import col, upper, sum as sum_

settlements = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIM_SETTLEMENTS"
)

payments = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.PREMIUM_PAYMENTS"
)

# Total net settlement
net_amount = settlements.select(
    sum_(col("NET_SETTLEMENT_AMOUNT")).alias("TOTAL")
).collect()[0]["TOTAL"] or 0

# Total premium collected
paid_payments = payments.filter(
    upper(col("PAYMENT_STATUS")) == "PAID"
)

premium_amount = paid_payments.select(
    sum_(col("PAYMENT_AMOUNT")).alias("TOTAL")
).collect()[0]["TOTAL"] or 0

# Calculate loss ratio
loss_ratio = (
    (float(net_amount) / float(premium_amount)) * 100
    if premium_amount > 0 else 0
)

print("KPI 9: LOSS RATIO PROXY")
print("Net Settlement Amount:", net_amount)
print("Premium Collected:", premium_amount)
print("Loss Ratio Proxy:", round(loss_ratio, 2), "%")

In [ ]:
from snowflake.snowpark.functions import col, upper

fraud = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.FRAUD_CASES"
)

open_cases = fraud.filter(
    upper(col("INVESTIGATION_STATUS")).isin(
        "OPEN",
        "IN_PROGRESS",
        "UNDER_INVESTIGATION"
    )
)

open_count = open_cases.count()

print("KPI 10: OPEN FRAUD INVESTIGATIONS")
print("Open Investigations:", open_count)

In [ ]:
fraud = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.FRAUD_CASES"
)

fraud.print_schema()

In [ ]:
from snowflake.snowpark.functions import col, upper

fraud = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.FRAUD_CASES"
)

# Get confirmed fraud cases
confirmed_cases = fraud.filter(
    upper(col("INVESTIGATION_OUTCOME")).isin(
        "CONFIRMED",
        "CONFIRMED_FRAUD"
    )
)

# Get completed investigations
closed_cases = fraud.filter(
    upper(col("INVESTIGATION_STATUS")).isin(
        "CLOSED",
        "COMPLETED",
        "RESOLVED"
    )
)

confirmed_count = confirmed_cases.count()
closed_count = closed_cases.count()

# Calculate confirmed fraud rate
confirmed_fraud_rate = (
    (confirmed_count / closed_count) * 100
    if closed_count > 0 else 0
)

print("KPI 11: CONFIRMED FRAUD RATE")
print("Confirmed Fraud Cases:", confirmed_count)
print("Closed Investigations:", closed_count)
print("Confirmed Fraud Rate:", round(confirmed_fraud_rate, 2), "%")

In [ ]:

session.sql("""
    CREATE STREAM IF NOT EXISTS
    INSURANCEPOLICY_LIFECYCLE_DB.SILVER.STREAM_CLAIMS_SCD2
    ON TABLE INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2
""").collect()

print("Silver Claims Stream created successfully!")

session.sql("""
    SHOW STREAMS IN SCHEMA
    INSURANCEPOLICY_LIFECYCLE_DB.SILVER
""").show()

In [ ]:
claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

claims.group_by("IS_CURRENT").count().show()

claims.select(
    "CLAIM_ID",
    "IS_CURRENT",
    "IS_DELETED",
    "SOURCE_UPDATED_TS",
    "_LOAD_TS"
).sort("_LOAD_TS", ascending=False).show(10)

In [ ]:
session.sql("""
    SHOW STREAMS IN SCHEMA
    INSURANCEPOLICY_LIFECYCLE_DB.SILVER
""").show()

session.sql("""
    SELECT SYSTEM$STREAM_HAS_DATA(
        'INSURANCEPOLICY_LIFECYCLE_DB.SILVER.STREAM_CLAIMS_SCD2'
    ) AS HAS_CHANGES
""").show()

claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

claims.filter(
    (col("CLAIM_ID") == "CLM0000001") &
    (col("IS_CURRENT") == True)
).show()

In [ ]:
stream = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.STREAM_CLAIMS_SCD2"
)

print("Stream record count:", stream.count())

stream.show()

In [ ]:

# ============================================================
# INSURANCE PROJECT - AUTOMATED FRAUD RISK CALCULATION
# SNOWPARK PYTHON WORKSHEET
# ============================================================

# STEP 1: CONNECT TO SNOWFLAKE

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, when, upper, max as max_,
    coalesce, lit, least, current_timestamp
)

session = get_active_session()

print("Connected to Snowflake successfully")


# STEP 2: READ SILVER TABLES

claims = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CLAIMS_SCD2"
)

fraud = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.FRAUD_CASES"
)

policies = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.POLICIES"
)

customers = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.SILVER.CUSTOMERS"
)

print("Claims count:", claims.count())
print("Fraud cases count:", fraud.count())
print("Policies count:", policies.count())
print("Customers count:", customers.count())


# STEP 3: SELECT CURRENT CLAIMS ONLY

current_claims = claims.filter(
    (col("IS_CURRENT") == True) &
    (col("IS_DELETED") == False)
)

print("Current claims count:", current_claims.count())


# STEP 4: PREPARE FRAUD DATA

fraud_summary = fraud.group_by("CLAIM_ID").agg(
    max_("FRAUD_SCORE").alias("EXISTING_FRAUD_SCORE")
)

customer_data = customers.select(
    "CUSTOMER_ID",
    "CUSTOMER_RISK_CATEGORY"
)

policy_data = policies.select(
    "POLICY_ID",
    "UNDERWRITING_RISK_BAND"
)


# STEP 5: JOIN ALL REQUIRED DATA

joined_data = (
    current_claims
    .join(customer_data, "CUSTOMER_ID", "left")
    .join(policy_data, "POLICY_ID", "left")
    .join(fraud_summary, "CLAIM_ID", "left")
)

joined_data = joined_data.with_column(
    "EXISTING_FRAUD_SCORE",
    coalesce(col("EXISTING_FRAUD_SCORE"), lit(0))
)

print("Joined data:")
joined_data.select(
    "CLAIM_ID",
    "CLAIM_AMOUNT",
    "CUSTOMER_RISK_CATEGORY",
    "UNDERWRITING_RISK_BAND",
    "EXISTING_FRAUD_SCORE"
).show(10)


# STEP 6: CALCULATE CLAIM RISK SCORE

scored_data = joined_data.with_column(
    "CLAIM_RISK_SCORE",
    when(col("CLAIM_AMOUNT") > 500000, 30)
    .when(col("CLAIM_AMOUNT") > 100000, 20)
    .when(col("CLAIM_AMOUNT") > 50000, 10)
    .otherwise(0)
)


# STEP 7: CALCULATE CUSTOMER RISK SCORE

scored_data = scored_data.with_column(
    "CUSTOMER_RISK_SCORE",
    when(
        upper(col("CUSTOMER_RISK_CATEGORY")).isin(
            "HIGH", "VERY_HIGH"
        ), 25
    )
    .when(
        upper(col("CUSTOMER_RISK_CATEGORY")) == "MEDIUM", 15
    )
    .otherwise(0)
)


# STEP 8: CALCULATE POLICY RISK SCORE

scored_data = scored_data.with_column(
    "POLICY_RISK_SCORE",
    when(
        upper(col("UNDERWRITING_RISK_BAND")).isin(
            "HIGH", "VERY_HIGH"
        ), 20
    )
    .when(
        upper(col("UNDERWRITING_RISK_BAND")) == "MEDIUM", 10
    )
    .otherwise(0)
)


# STEP 9: CALCULATE FRAUD HISTORY SCORE

scored_data = scored_data.with_column(
    "FRAUD_HISTORY_SCORE",
    when(col("EXISTING_FRAUD_SCORE") >= 70, 25)
    .when(col("EXISTING_FRAUD_SCORE") >= 40, 15)
    .otherwise(0)
)


# STEP 10: CALCULATE TOTAL RISK SCORE

scored_data = scored_data.with_column(
    "TOTAL_RISK_SCORE",
    least(
        col("CLAIM_RISK_SCORE") +
        col("CUSTOMER_RISK_SCORE") +
        col("POLICY_RISK_SCORE") +
        col("FRAUD_HISTORY_SCORE"),
        lit(100)
    )
)


# STEP 11: ASSIGN RISK CATEGORY

scored_data = scored_data.with_column(
    "RISK_CATEGORY",
    when(col("TOTAL_RISK_SCORE") >= 70, "HIGH")
    .when(col("TOTAL_RISK_SCORE") >= 40, "MEDIUM")
    .otherwise("LOW")
)


# STEP 12: PREPARE FINAL OUTPUT

final_scores = scored_data.select(
    col("CLAIM_ID"),
    col("CLAIM_RISK_SCORE"),
    col("CUSTOMER_RISK_SCORE"),
    col("POLICY_RISK_SCORE"),
    col("FRAUD_HISTORY_SCORE"),
    col("TOTAL_RISK_SCORE"),
    col("RISK_CATEGORY"),
    current_timestamp().alias("SCORING_TIMESTAMP")
)

print("Final fraud risk calculation:")
final_scores.show(10)


# STEP 13: CREATE GOLD SCHEMA AND TABLE

session.sql("""
    CREATE SCHEMA IF NOT EXISTS
    INSURANCEPOLICY_LIFECYCLE_DB.GOLD
""").collect()

session.sql("""
    CREATE TABLE IF NOT EXISTS
    INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES
    (
        CLAIM_ID VARCHAR(50),
        CLAIM_RISK_SCORE NUMBER(5,2),
        CUSTOMER_RISK_SCORE NUMBER(5,2),
        POLICY_RISK_SCORE NUMBER(5,2),
        FRAUD_HISTORY_SCORE NUMBER(5,2),
        TOTAL_RISK_SCORE NUMBER(5,2),
        RISK_CATEGORY VARCHAR(20),
        SCORING_TIMESTAMP TIMESTAMP_NTZ
    )
""").collect()


# STEP 14: LOAD DATA INTO GOLD TABLE

final_scores.write.mode("overwrite").save_as_table(
    "INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES"
)

print("Fraud risk scores loaded successfully!")


# STEP 15: VERIFY GOLD TABLE

gold_scores = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES"
)

print("Total scored claims:", gold_scores.count())

print("Risk category distribution:")
gold_scores.group_by("RISK_CATEGORY").count().show()

print("Final Gold fraud risk records:")
gold_scores.show(10)

print("FRAUD RISK CALCULATION COMPLETED SUCCESSFULLY")


In [ ]:
session.sql("""
    USE DATABASE INSURANCEPOLICY_LIFECYCLE_DB
""").collect()

session.sql("""
    USE SCHEMA SILVER
""").collect()

session.sql("""
    USE WAREHOUSE COMPUTE_WH
""").collect()

print("Database, schema and warehouse selected successfully!")


session.sql("""
    SELECT
        CURRENT_DATABASE(),
        CURRENT_SCHEMA(),
        CURRENT_WAREHOUSE(),
        CURRENT_ROLE()
""").show()



# ============================================================
# INCREMENTAL FRAUD RISK SCORING USING SILVER STREAM
# ============================================================

from snowflake.snowpark.functions import (
    col, when, upper, max as max_,
    coalesce, lit, least, current_timestamp
)

# STEP 1: DEFINE TABLES

db = "INSURANCEPOLICY_LIFECYCLE_DB"

stream_name = db + ".SILVER.STREAM_CLAIMS_SCD2"
claims_name = db + ".SILVER.CLAIMS_SCD2"
fraud_name = db + ".SILVER.FRAUD_CASES"
policies_name = db + ".SILVER.POLICIES"
customers_name = db + ".SILVER.CUSTOMERS"
gold_name = db + ".GOLD.FRAUD_RISK_SCORES"

# STEP 2: CREATE TEMPORARY TABLES

session.sql("""
CREATE TEMP TABLE IF NOT EXISTS TMP_CHANGED_CLAIMS
(
    CLAIM_ID VARCHAR(50)
)
""").collect()

session.sql("""
CREATE TEMP TABLE IF NOT EXISTS TMP_INCREMENTAL_SCORES
(
    CLAIM_ID VARCHAR(50),
    CLAIM_RISK_SCORE NUMBER(5,2),
    CUSTOMER_RISK_SCORE NUMBER(5,2),
    POLICY_RISK_SCORE NUMBER(5,2),
    FRAUD_HISTORY_SCORE NUMBER(5,2),
    TOTAL_RISK_SCORE NUMBER(5,2),
    RISK_CATEGORY VARCHAR(20),
    SCORING_TIMESTAMP TIMESTAMP_NTZ
)
""").collect()

# STEP 3: BEGIN TRANSACTION

session.sql("BEGIN TRANSACTION").collect()

try:

    # Clear temporary data from previous executions
    session.sql("DELETE FROM TMP_CHANGED_CLAIMS").collect()
    session.sql("DELETE FROM TMP_INCREMENTAL_SCORES").collect()

    # STEP 4: CAPTURE CHANGED CLAIM IDS FROM STREAM

    session.sql("""
        INSERT INTO TMP_CHANGED_CLAIMS
        SELECT DISTINCT CLAIM_ID
        FROM """ + stream_name + """
        WHERE CLAIM_ID IS NOT NULL
    """).collect()

    changed_count = session.table(
        "TMP_CHANGED_CLAIMS"
    ).count()

    print("Changed claims:", changed_count)

    if changed_count == 0:

        session.sql("COMMIT").collect()
        print("No new changes found. No scoring required.")

    else:

        # STEP 5: READ CURRENT CLAIMS

        claims = session.table(claims_name)

        current_claims = claims.filter(
            (col("IS_CURRENT") == True) &
            (col("IS_DELETED") == False)
        )

        changed_ids = session.table(
            "TMP_CHANGED_CLAIMS"
        ).select("CLAIM_ID").distinct()

        # Identify affected customers

        affected_claims = current_claims.join(
            changed_ids,
            "CLAIM_ID",
            "inner"
        )

        affected_customers = affected_claims.select(
            "CUSTOMER_ID"
        ).distinct()

        # Recalculate all current claims for those customers

        all_affected_claims = current_claims.join(
            affected_customers,
            "CUSTOMER_ID",
            "inner"
        )

        print(
            "Claims selected for recalculation:",
            all_affected_claims.count()
        )

        # STEP 6: READ OTHER SILVER TABLES

        fraud = session.table(fraud_name)
        policies = session.table(policies_name)
        customers = session.table(customers_name)

        fraud_summary = fraud.group_by("CLAIM_ID").agg(
            max_("FRAUD_SCORE").alias("EXISTING_FRAUD_SCORE")
        )

        customer_data = customers.select(
            "CUSTOMER_ID",
            "CUSTOMER_RISK_CATEGORY"
        )

        policy_data = policies.select(
            "POLICY_ID",
            "UNDERWRITING_RISK_BAND"
        )

        # STEP 7: JOIN DATA

        joined_data = (
            all_affected_claims
            .join(customer_data, "CUSTOMER_ID", "left")
            .join(policy_data, "POLICY_ID", "left")
            .join(fraud_summary, "CLAIM_ID", "left")
        )

        joined_data = joined_data.with_column(
            "EXISTING_FRAUD_SCORE",
            coalesce(col("EXISTING_FRAUD_SCORE"), lit(0))
        )

        # STEP 8: CLAIM RISK SCORE

        scored_data = joined_data.with_column(
            "CLAIM_RISK_SCORE",
            when(col("CLAIM_AMOUNT") > 500000, 30)
            .when(col("CLAIM_AMOUNT") > 100000, 20)
            .when(col("CLAIM_AMOUNT") > 50000, 10)
            .otherwise(0)
        )

        # STEP 9: CUSTOMER RISK SCORE

        scored_data = scored_data.with_column(
            "CUSTOMER_RISK_SCORE",
            when(
                upper(col("CUSTOMER_RISK_CATEGORY")).isin(
                    "HIGH", "VERY_HIGH"
                ), 25
            )
            .when(
                upper(col("CUSTOMER_RISK_CATEGORY")) == "MEDIUM", 15
            )
            .otherwise(0)
        )

        # STEP 10: POLICY RISK SCORE

        scored_data = scored_data.with_column(
            "POLICY_RISK_SCORE",
            when(
                upper(col("UNDERWRITING_RISK_BAND")).isin(
                    "HIGH", "VERY_HIGH"
                ), 20
            )
            .when(
                upper(col("UNDERWRITING_RISK_BAND")) == "MEDIUM", 10
            )
            .otherwise(0)
        )

        # STEP 11: FRAUD HISTORY SCORE

        scored_data = scored_data.with_column(
            "FRAUD_HISTORY_SCORE",
            when(col("EXISTING_FRAUD_SCORE") >= 70, 25)
            .when(col("EXISTING_FRAUD_SCORE") >= 40, 15)
            .otherwise(0)
        )

        # STEP 12: TOTAL SCORE

        scored_data = scored_data.with_column(
            "TOTAL_RISK_SCORE",
            least(
                col("CLAIM_RISK_SCORE") +
                col("CUSTOMER_RISK_SCORE") +
                col("POLICY_RISK_SCORE") +
                col("FRAUD_HISTORY_SCORE"),
                lit(100)
            )
        )

        # STEP 13: RISK CATEGORY

        scored_data = scored_data.with_column(
            "RISK_CATEGORY",
            when(col("TOTAL_RISK_SCORE") >= 70, "HIGH")
            .when(col("TOTAL_RISK_SCORE") >= 40, "MEDIUM")
            .otherwise("LOW")
        )

        # STEP 14: PREPARE FINAL SCORES

        final_scores = scored_data.select(
            col("CLAIM_ID"),
            col("CLAIM_RISK_SCORE"),
            col("CUSTOMER_RISK_SCORE"),
            col("POLICY_RISK_SCORE"),
            col("FRAUD_HISTORY_SCORE"),
            col("TOTAL_RISK_SCORE"),
            col("RISK_CATEGORY"),
            current_timestamp().alias("SCORING_TIMESTAMP")
        ).distinct()

        print("New incremental scores:")
        final_scores.show(10)

        # STEP 15: STORE SCORES IN TEMP TABLE

        final_scores.write.mode("append").save_as_table(
            "TMP_INCREMENTAL_SCORES"
        )

        # STEP 16: MERGE INTO GOLD TABLE

        session.sql("""
            MERGE INTO """ + gold_name + """ T
            USING TMP_INCREMENTAL_SCORES S
            ON T.CLAIM_ID = S.CLAIM_ID

            WHEN MATCHED THEN UPDATE SET
                T.CLAIM_RISK_SCORE = S.CLAIM_RISK_SCORE,
                T.CUSTOMER_RISK_SCORE = S.CUSTOMER_RISK_SCORE,
                T.POLICY_RISK_SCORE = S.POLICY_RISK_SCORE,
                T.FRAUD_HISTORY_SCORE = S.FRAUD_HISTORY_SCORE,
                T.TOTAL_RISK_SCORE = S.TOTAL_RISK_SCORE,
                T.RISK_CATEGORY = S.RISK_CATEGORY,
                T.SCORING_TIMESTAMP = S.SCORING_TIMESTAMP

            WHEN NOT MATCHED THEN INSERT
            (
                CLAIM_ID,
                CLAIM_RISK_SCORE,
                CUSTOMER_RISK_SCORE,
                POLICY_RISK_SCORE,
                FRAUD_HISTORY_SCORE,
                TOTAL_RISK_SCORE,
                RISK_CATEGORY,
                SCORING_TIMESTAMP
            )
            VALUES
            (
                S.CLAIM_ID,
                S.CLAIM_RISK_SCORE,
                S.CUSTOMER_RISK_SCORE,
                S.POLICY_RISK_SCORE,
                S.FRAUD_HISTORY_SCORE,
                S.TOTAL_RISK_SCORE,
                S.RISK_CATEGORY,
                S.SCORING_TIMESTAMP
            )
        """).collect()

        # STEP 17: COMMIT TRANSACTION

        session.sql("COMMIT").collect()

        print("Incremental fraud scoring completed successfully!")

except Exception as e:

    session.sql("ROLLBACK").collect()

    print("Fraud scoring failed. Transaction rolled back.")
    raise e


# STEP 18: VERIFY GOLD OUTPUT

gold_scores = session.table(gold_name)

print("Total records in Gold:", gold_scores.count())

gold_scores.group_by("RISK_CATEGORY").count().show()

gold_scores.show(10)



In [ ]:
policies.group_by(
    "UNDERWRITING_RISK_BAND"
).count().show()

In [ ]:
scored_data = scored_data.with_column(
    "POLICY_RISK_SCORE",
    when(upper(col("UNDERWRITING_RISK_BAND")) == "C", 20)
    .when(upper(col("UNDERWRITING_RISK_BAND")) == "B", 10)
    .when(upper(col("UNDERWRITING_RISK_BAND")) == "A", 0)
    .otherwise(0)
)

print("Policy risk scores updated")

In [ ]:
scored_data = scored_data.with_column(
    "TOTAL_RISK_SCORE",
    least(
        col("CLAIM_RISK_SCORE") +
        col("CUSTOMER_RISK_SCORE") +
        col("POLICY_RISK_SCORE") +
        col("FRAUD_HISTORY_SCORE"),
        lit(100)
    )
)

scored_data = scored_data.with_column(
    "RISK_CATEGORY",
    when(col("TOTAL_RISK_SCORE") >= 70, "HIGH")
    .when(col("TOTAL_RISK_SCORE") >= 40, "MEDIUM")
    .otherwise("LOW")
)

scored_data.select(
    "CLAIM_ID",
    "POLICY_RISK_SCORE",
    "TOTAL_RISK_SCORE",
    "RISK_CATEGORY"
).show(10)

In [ ]:
final_scores = scored_data.select(
    col("CLAIM_ID"),
    col("CLAIM_RISK_SCORE"),
    col("CUSTOMER_RISK_SCORE"),
    col("POLICY_RISK_SCORE"),
    col("FRAUD_HISTORY_SCORE"),
    col("TOTAL_RISK_SCORE"),
    col("RISK_CATEGORY"),
    current_timestamp().alias("SCORING_TIMESTAMP")
)

final_scores.write.mode("overwrite").save_as_table(
    "INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES"
)

print("Gold fraud risk scores refreshed successfully!")

In [ ]:
gold_scores = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES"
)

gold_scores.group_by("RISK_CATEGORY").count().show()
gold_scores.show(10)

In [ ]:
session.sql("""
    CREATE STAGE IF NOT EXISTS
    INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_PROCEDURE_STAGE
""").collect()

print("Procedure stage created successfully")

In [ ]:

from snowflake.snowpark.functions import (
    col,
    when,
    upper,
    max as max_,
    coalesce,
    lit,
    least,
    current_timestamp
)

def calculate_fraud_risk(session):

    db = "INSURANCEPOLICY_LIFECYCLE_DB"

    # -----------------------------------------
    # 1. READ SILVER TABLES
    # -----------------------------------------

    claims = session.table(
        db + ".SILVER.CLAIMS_SCD2"
    )

    fraud = session.table(
        db + ".SILVER.FRAUD_CASES"
    )

    policies = session.table(
        db + ".SILVER.POLICIES"
    )

    customers = session.table(
        db + ".SILVER.CUSTOMERS"
    )

    # -----------------------------------------
    # 2. SELECT CURRENT CLAIMS
    # -----------------------------------------

    current_claims = claims.filter(
        (col("IS_CURRENT") == True) &
        (col("IS_DELETED") == False)
    )

    # -----------------------------------------
    # 3. PREPARE FRAUD HISTORY
    # -----------------------------------------

    fraud_summary = fraud.group_by(
        "CLAIM_ID"
    ).agg(
        max_("FRAUD_SCORE").alias(
            "EXISTING_FRAUD_SCORE"
        )
    )

    customer_data = customers.select(
        "CUSTOMER_ID",
        "CUSTOMER_RISK_CATEGORY"
    )

    policy_data = policies.select(
        "POLICY_ID",
        "UNDERWRITING_RISK_BAND"
    )

    # -----------------------------------------
    # 4. JOIN THE TABLES
    # -----------------------------------------

    joined_data = (
        current_claims
        .join(customer_data, "CUSTOMER_ID", "left")
        .join(policy_data, "POLICY_ID", "left")
        .join(fraud_summary, "CLAIM_ID", "left")
    )

    joined_data = joined_data.with_column(
        "EXISTING_FRAUD_SCORE",
        coalesce(
            col("EXISTING_FRAUD_SCORE"),
            lit(0)
        )
    )

    # -----------------------------------------
    # 5. CLAIM RISK SCORE
    # -----------------------------------------

    scored_data = joined_data.with_column(
        "CLAIM_RISK_SCORE",
        when(col("CLAIM_AMOUNT") > 500000, 30)
        .when(col("CLAIM_AMOUNT") > 100000, 20)
        .when(col("CLAIM_AMOUNT") > 50000, 10)
        .otherwise(0)
    )

    # -----------------------------------------
    # 6. CUSTOMER RISK SCORE
    # -----------------------------------------

    scored_data = scored_data.with_column(
        "CUSTOMER_RISK_SCORE",
        when(
            upper(col("CUSTOMER_RISK_CATEGORY"))
            .isin("HIGH", "VERY_HIGH"), 25
        )
        .when(
            upper(col("CUSTOMER_RISK_CATEGORY")) == "MEDIUM",
            15
        )
        .otherwise(0)
    )

    # -----------------------------------------
    # 7. POLICY RISK SCORE
    # PROVISIONAL A/B/C MAPPING
    # -----------------------------------------

    scored_data = scored_data.with_column(
        "POLICY_RISK_SCORE",
        when(
            upper(col("UNDERWRITING_RISK_BAND")) == "C", 20
        )
        .when(
            upper(col("UNDERWRITING_RISK_BAND")) == "B", 10
        )
        .when(
            upper(col("UNDERWRITING_RISK_BAND")) == "A", 0
        )
        .otherwise(0)
    )

    # -----------------------------------------
    # 8. FRAUD HISTORY SCORE
    # -----------------------------------------

    scored_data = scored_data.with_column(
        "FRAUD_HISTORY_SCORE",
        when(col("EXISTING_FRAUD_SCORE") >= 70, 25)
        .when(col("EXISTING_FRAUD_SCORE") >= 40, 15)
        .otherwise(0)
    )

    # -----------------------------------------
    # 9. TOTAL RISK SCORE
    # -----------------------------------------

    scored_data = scored_data.with_column(
        "TOTAL_RISK_SCORE",
        least(
            col("CLAIM_RISK_SCORE") +
            col("CUSTOMER_RISK_SCORE") +
            col("POLICY_RISK_SCORE") +
            col("FRAUD_HISTORY_SCORE"),
            lit(100)
        )
    )

    # -----------------------------------------
    # 10. RISK CATEGORY
    # -----------------------------------------

    scored_data = scored_data.with_column(
        "RISK_CATEGORY",
        when(col("TOTAL_RISK_SCORE") >= 70, "HIGH")
        .when(col("TOTAL_RISK_SCORE") >= 40, "MEDIUM")
        .otherwise("LOW")
    )

    # -----------------------------------------
    # 11. PREPARE FINAL DATA
    # -----------------------------------------

    final_scores = scored_data.select(
        col("CLAIM_ID"),
        col("CLAIM_RISK_SCORE"),
        col("CUSTOMER_RISK_SCORE"),
        col("POLICY_RISK_SCORE"),
        col("FRAUD_HISTORY_SCORE"),
        col("TOTAL_RISK_SCORE"),
        col("RISK_CATEGORY"),
        current_timestamp().alias("SCORING_TIMESTAMP")
    )

    # -----------------------------------------
    # 12. SAVE SCORES TO TEMPORARY STAGING TABLE
    # -----------------------------------------

    temp_table = (
        db + ".GOLD.TMP_FRAUD_RISK_SCORES_STAGE"
    )

    final_scores.write.mode("overwrite").save_as_table(
        temp_table,
        table_type="temporary"
    )

    scored_count = session.table(temp_table).count()

    # -----------------------------------------
    # 13. MERGE INTO GOLD TABLE
    # -----------------------------------------

    target_table = db + ".GOLD.FRAUD_RISK_SCORES"

    session.sql("""
        MERGE INTO """ + target_table + """ T
        USING """ + temp_table + """ S
        ON T.CLAIM_ID = S.CLAIM_ID

        WHEN MATCHED THEN UPDATE SET
            T.CLAIM_RISK_SCORE = S.CLAIM_RISK_SCORE,
            T.CUSTOMER_RISK_SCORE = S.CUSTOMER_RISK_SCORE,
            T.POLICY_RISK_SCORE = S.POLICY_RISK_SCORE,
            T.FRAUD_HISTORY_SCORE = S.FRAUD_HISTORY_SCORE,
            T.TOTAL_RISK_SCORE = S.TOTAL_RISK_SCORE,
            T.RISK_CATEGORY = S.RISK_CATEGORY,
            T.SCORING_TIMESTAMP = S.SCORING_TIMESTAMP

        WHEN NOT MATCHED THEN INSERT
        (
            CLAIM_ID,
            CLAIM_RISK_SCORE,
            CUSTOMER_RISK_SCORE,
            POLICY_RISK_SCORE,
            FRAUD_HISTORY_SCORE,
            TOTAL_RISK_SCORE,
            RISK_CATEGORY,
            SCORING_TIMESTAMP
        )
        VALUES
        (
            S.CLAIM_ID,
            S.CLAIM_RISK_SCORE,
            S.CUSTOMER_RISK_SCORE,
            S.POLICY_RISK_SCORE,
            S.FRAUD_HISTORY_SCORE,
            S.TOTAL_RISK_SCORE,
            S.RISK_CATEGORY,
            S.SCORING_TIMESTAMP
        )
    """).collect()

    # -----------------------------------------
    # 14. RETURN EXECUTION RESULT
    # -----------------------------------------

    return (
        "SUCCESS: Fraud risk calculation completed. "
        + str(scored_count)
        + " current claims processed and Gold table merged."
    )


In [ ]:
from snowflake.snowpark.types import StringType

procedure = session.sproc.register(
    func=calculate_fraud_risk,
    return_type=StringType(),
    input_types=[],
    name="INSURANCEPOLICY_LIFECYCLE_DB.GOLD.CALCULATE_FRAUD_RISK",
    is_permanent=True,
    stage_location="@INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_PROCEDURE_STAGE",
    packages=["snowflake-snowpark-python"],
    replace=True
)

print("Permanent Stored Procedure created successfully!")
print(procedure.name)

In [ ]:
from snowflake.snowpark.functions import (
    when_matched,
    when_not_matched
)

In [ ]:

# ============================================================
# INSURANCE PROJECT - FRAUD RISK STORED PROCEDURE
# SNOWPARK PYTHON
# ============================================================

from snowflake.snowpark.context import get_active_session

from snowflake.snowpark.functions import (
    col,
    when,
    upper,
    max as max_,
    coalesce,
    lit,
    least,
    current_timestamp,
    when_matched,
    when_not_matched
)

from snowflake.snowpark.types import StringType

# STEP 1: CONNECT TO SNOWFLAKE

session = get_active_session()

session.sql("""
    USE DATABASE INSURANCEPOLICY_LIFECYCLE_DB
""").collect()

session.sql("""
    USE SCHEMA GOLD
""").collect()

session.sql("""
    USE WAREHOUSE COMPUTE_WH
""").collect()

print("Snowflake connection successful")


# STEP 2: DEFINE THE STORED PROCEDURE FUNCTION

def calculate_fraud_risk(session):

    db = "INSURANCEPOLICY_LIFECYCLE_DB"

    # ----------------------------------------
    # STEP 3: READ SILVER TABLES
    # ----------------------------------------

    claims = session.table(
        db + ".SILVER.CLAIMS_SCD2"
    )

    fraud = session.table(
        db + ".SILVER.FRAUD_CASES"
    )

    policies = session.table(
        db + ".SILVER.POLICIES"
    )

    customers = session.table(
        db + ".SILVER.CUSTOMERS"
    )

    # ----------------------------------------
    # STEP 4: SELECT CURRENT CLAIMS
    # ----------------------------------------

    current_claims = claims.filter(
        (col("IS_CURRENT") == True) &
        (col("IS_DELETED") == False)
    )

    # ----------------------------------------
    # STEP 5: PREPARE FRAUD HISTORY
    # ----------------------------------------

    fraud_summary = fraud.group_by(
        "CLAIM_ID"
    ).agg(
        max_("FRAUD_SCORE").alias(
            "EXISTING_FRAUD_SCORE"
        )
    )

    customer_data = customers.select(
        "CUSTOMER_ID",
        "CUSTOMER_RISK_CATEGORY"
    )

    policy_data = policies.select(
        "POLICY_ID",
        "UNDERWRITING_RISK_BAND"
    )

    # ----------------------------------------
    # STEP 6: JOIN ALL TABLES
    # ----------------------------------------

    joined_data = (
        current_claims
        .join(customer_data, "CUSTOMER_ID", "left")
        .join(policy_data, "POLICY_ID", "left")
        .join(fraud_summary, "CLAIM_ID", "left")
    )

    joined_data = joined_data.with_column(
        "EXISTING_FRAUD_SCORE",
        coalesce(
            col("EXISTING_FRAUD_SCORE"),
            lit(0)
        )
    )

    # ----------------------------------------
    # STEP 7: CLAIM RISK SCORE
    # ----------------------------------------

    scored_data = joined_data.with_column(
        "CLAIM_RISK_SCORE",
        when(col("CLAIM_AMOUNT") > 500000, 30)
        .when(col("CLAIM_AMOUNT") > 100000, 20)
        .when(col("CLAIM_AMOUNT") > 50000, 10)
        .otherwise(0)
    )

    # ----------------------------------------
    # STEP 8: CUSTOMER RISK SCORE
    # ----------------------------------------

    scored_data = scored_data.with_column(
        "CUSTOMER_RISK_SCORE",
        when(
            upper(col("CUSTOMER_RISK_CATEGORY"))
            .isin("HIGH", "VERY_HIGH"), 25
        )
        .when(
            upper(col("CUSTOMER_RISK_CATEGORY")) == "MEDIUM",
            15
        )
        .otherwise(0)
    )

    # ----------------------------------------
    # STEP 9: POLICY RISK SCORE
    # A = 0, B = 10, C = 20
    # PROVISIONAL BUSINESS MAPPING
    # ----------------------------------------

    scored_data = scored_data.with_column(
        "POLICY_RISK_SCORE",
        when(
            upper(col("UNDERWRITING_RISK_BAND")) == "C", 20
        )
        .when(
            upper(col("UNDERWRITING_RISK_BAND")) == "B", 10
        )
        .when(
            upper(col("UNDERWRITING_RISK_BAND")) == "A", 0
        )
        .otherwise(0)
    )

    # ----------------------------------------
    # STEP 10: FRAUD HISTORY SCORE
    # ----------------------------------------

    scored_data = scored_data.with_column(
        "FRAUD_HISTORY_SCORE",
        when(col("EXISTING_FRAUD_SCORE") >= 70, 25)
        .when(col("EXISTING_FRAUD_SCORE") >= 40, 15)
        .otherwise(0)
    )

    # ----------------------------------------
    # STEP 11: TOTAL RISK SCORE
    # ----------------------------------------

    scored_data = scored_data.with_column(
        "TOTAL_RISK_SCORE",
        least(
            col("CLAIM_RISK_SCORE") +
            col("CUSTOMER_RISK_SCORE") +
            col("POLICY_RISK_SCORE") +
            col("FRAUD_HISTORY_SCORE"),
            lit(100)
        )
    )

    # ----------------------------------------
    # STEP 12: RISK CATEGORY
    # ----------------------------------------

    scored_data = scored_data.with_column(
        "RISK_CATEGORY",
        when(col("TOTAL_RISK_SCORE") >= 70, "HIGH")
        .when(col("TOTAL_RISK_SCORE") >= 40, "MEDIUM")
        .otherwise("LOW")
    )

    # ----------------------------------------
    # STEP 13: PREPARE FINAL SCORES
    # ----------------------------------------

    final_scores = scored_data.select(
        col("CLAIM_ID"),
        col("CLAIM_RISK_SCORE"),
        col("CUSTOMER_RISK_SCORE"),
        col("POLICY_RISK_SCORE"),
        col("FRAUD_HISTORY_SCORE"),
        col("TOTAL_RISK_SCORE"),
        col("RISK_CATEGORY"),
        current_timestamp().alias("SCORING_TIMESTAMP")
    )

    # ----------------------------------------
    # STEP 14: COUNT SCORED CLAIMS
    # ----------------------------------------

    scored_count = final_scores.count()

    # ----------------------------------------
    # STEP 15: MERGE INTO GOLD TABLE
    # ----------------------------------------

    target_table = session.table(
        db + ".GOLD.FRAUD_RISK_SCORES"
    )

    merge_result = target_table.merge(
        final_scores,
        target_table["CLAIM_ID"] == final_scores["CLAIM_ID"],
        [
            when_matched().update({
                "CLAIM_RISK_SCORE":
                    final_scores["CLAIM_RISK_SCORE"],

                "CUSTOMER_RISK_SCORE":
                    final_scores["CUSTOMER_RISK_SCORE"],

                "POLICY_RISK_SCORE":
                    final_scores["POLICY_RISK_SCORE"],

                "FRAUD_HISTORY_SCORE":
                    final_scores["FRAUD_HISTORY_SCORE"],

                "TOTAL_RISK_SCORE":
                    final_scores["TOTAL_RISK_SCORE"],

                "RISK_CATEGORY":
                    final_scores["RISK_CATEGORY"],

                "SCORING_TIMESTAMP":
                    final_scores["SCORING_TIMESTAMP"]
            }),

            when_not_matched().insert({
                "CLAIM_ID":
                    final_scores["CLAIM_ID"],

                "CLAIM_RISK_SCORE":
                    final_scores["CLAIM_RISK_SCORE"],

                "CUSTOMER_RISK_SCORE":
                    final_scores["CUSTOMER_RISK_SCORE"],

                "POLICY_RISK_SCORE":
                    final_scores["POLICY_RISK_SCORE"],

                "FRAUD_HISTORY_SCORE":
                    final_scores["FRAUD_HISTORY_SCORE"],

                "TOTAL_RISK_SCORE":
                    final_scores["TOTAL_RISK_SCORE"],

                "RISK_CATEGORY":
                    final_scores["RISK_CATEGORY"],

                "SCORING_TIMESTAMP":
                    final_scores["SCORING_TIMESTAMP"]
            })
        ]
    )

    # ----------------------------------------
    # STEP 16: RETURN RESULT
    # ----------------------------------------

    return (
        "SUCCESS: Fraud risk calculation completed. "
        + str(scored_count)
        + " current claims processed and Gold table merged."
    )


# STEP 17: REGISTER PERMANENT STORED PROCEDURE

procedure = session.sproc.register(
    func=calculate_fraud_risk,
    return_type=StringType(),
    input_types=[],
    name="INSURANCEPOLICY_LIFECYCLE_DB.GOLD.CALCULATE_FRAUD_RISK",
    is_permanent=True,
    stage_location="@INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_PROCEDURE_STAGE",
    packages=["snowflake-snowpark-python"],
    replace=True
)

print("Permanent Stored Procedure registered successfully!")
print(procedure.name)


In [ ]:
result = session.sql("""
    CALL INSURANCEPOLICY_LIFECYCLE_DB.GOLD.CALCULATE_FRAUD_RISK()
""").collect()

print(result[0][0])

In [ ]:
gold_scores = session.table(
    "INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES"
)

print("Total Gold records:", gold_scores.count())

gold_scores.group_by(
    "RISK_CATEGORY"
).count().show()

gold_scores.show(10)

In [ ]:
session.sql("""
    CALL INSURANCEPOLICY_LIFECYCLE_DB.GOLD.CALCULATE_FRAUD_RISK()
""").show()

print(
    "Distinct claims:",
    gold_scores.select("CLAIM_ID").distinct().count()
)

In [ ]:
%%sql -r dataframe_1
SHOW PROCEDURES LIKE 'CALCULATE_FRAUD_RISK'
IN SCHEMA INSURANCEPOLICY_LIFECYCLE_DB.GOLD;

In [ ]:
%%sql -r dataframe_2
SELECT COUNT(*)
FROM INSURANCEPOLICY_LIFECYCLE_DB.GOLD.FRAUD_RISK_SCORES;